# Experiment 1: Time and Frequency Domain, and the Pendulum
**Author:** Srishti Batham (500124490), Batch B2
CSAI4014 Algorithms for Intelligent Systems and Robotics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Signal definition
fs = 500.0  # Sampling rate in Hz
T = 2.0     # Duration in seconds
t = np.arange(0, T, 1.0/fs)
N = len(t)
rng = np.random.default_rng(42)

# Multi-tone signal
x = (1.00 * np.sin(2 * np.pi * 5 * t) + 
     0.50 * np.sin(2 * np.pi * 20 * t) + 
     0.25 * np.sin(2 * np.pi * 50 * t) + 
     0.20 * rng.standard_normal(N))

# Fourier Transform
X = np.fft.rfft(x)
f = np.fft.rfftfreq(N, d=1.0/fs)
amp = np.abs(X) * 2.0 / N
amp[0] /= 2.0

print(f"Sampling parameters: fs={fs} Hz, T={T} s, N={N}, df={fs/N} Hz, Nyquist={fs/2} Hz")
for freq in [5, 20, 50]:
    idx = np.argmin(np.abs(f - freq))
    print(f"Component {freq} Hz -> Measured freq: {f[idx]:.1f} Hz, Measured amp: {amp[idx]:.3f}")


In [ ]:
# Pendulum simulation
g, L = 9.81, 1.0
T0 = 2 * np.pi * np.sqrt(L / g)

def pendulum_deriv(t, y, b):
    theta, omega = y
    return [omega, -b * omega - (g / L) * np.sin(theta)]

angles = [5, 45, 95, 135, 175]
print(f"Analytical small-angle period T0 = {T0:.4f} s")
for ang in angles:
    sol = solve_ivp(pendulum_deriv, (0, 60), [np.radians(ang), 0.0], args=(0.0,), max_step=0.01, rtol=1e-8)
    th = sol.y[0]
    zc = [sol.t[i] - th[i]*(sol.t[i+1]-sol.t[i])/(th[i+1]-th[i]) for i in range(len(th)-1) if th[i]<=0 and th[i+1]>0]
    T_meas = np.mean(np.diff(zc))
    print(f"Angle {ang:3d} deg: Period = {T_meas:.4f} s (Ratio to T0 = {T_meas/T0:.2f})")
